# Knee v6: CoAtNet fold-0 screen (cache stays 160 px), with precheck + watchdog + epoch-time budget
Why: the public 0.949 write-up uses a CoAtNet + attention pooling and gets ~0.94 from the 4,349 competition studies alone. ConvNeXt-T here is 0.833 fold-0 macro and every ConvNeXt/DINO variant we screened is within noise of it.
Backbone: `coatnet_rmlp_1_rw2_224.sw_in12k_ft_in1k` built for 160 px (patched model.py, knee_pipeline_v6_5). GPU 0: lr_backbone 4e-5 | GPU 1: lr_backbone 2e-5. Same effective batch as the ConvNeXt runs but split finer for memory (`--bs 2 --accum 4`; the first try at bs 4 ran out of GPU memory), `--head tokens`, fold 0, 12 epochs.
Reference ConvNeXt-T fold-0 val macro (raw): ep1 0.700 | ep3 0.763 | ep5 0.800 | ep9 0.828 | ep12 0.832 (~450 s/epoch). Seed noise ~0.01: act only on a gain above ~+0.015.
SAFETY: (1) cell 2 builds the model WITH pretrained weights, runs forward/backward on CPU, then a GPU memory probe at the real training batch (fails if peak > 12.5 GB), before any job starts; (2) an arm with no epoch line after 45 min, or a stall of 45 min, is killed; (3) an arm whose FIRST epoch takes longer than 1,100 s is killed (12 epochs would exceed ~3.7 h), and its epoch time is printed so we can pick a smaller CoAtNet.
Inputs: competition data, `knee-mri-pipeline` (knee_pipeline_v6_5 with the CoAtNet-patched model.py), caching output, training output (oof_all.csv), the ConvNeXt 5-fold output. GPU T4 x2, internet ON, Save & Run All.

In [1]:
import os, sys, re, json, subprocess, time, threading
from pathlib import Path
import numpy as np, pandas as pd

COAT = "coatnet_rmlp_1_rw2_224.sw_in12k_ft_in1k"
W = Path(os.environ.get("KNEE_WORK", "/kaggle/working"))
CODE   = Path(os.environ.get("KNEE_CODE",   "/kaggle/input/datasets/theokorir/knee-mri-pipeline/knee_pipeline"))
DATA   = Path(os.environ.get("KNEE_DATA",   "/kaggle/input/competitions/rsna-knee-abnormality-detection"))
CACHE  = Path(os.environ.get("KNEE_CACHE",  "/kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/cache"))
LABELS = Path(os.environ.get("KNEE_LABELS", "/kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/labels_soft.csv"))
BASE   = Path(os.environ.get("KNEE_BASE",   "/kaggle/input/notebooks/theokorir/knee-mri-diagnosis-training/oof_all.csv"))
for name, p in dict(CODE=CODE, DATA=DATA, CACHE=CACHE, LABELS=LABELS, BASE=BASE).items():
    assert p.exists(), f"{name} not found: {p}"
for f in ("geom.py", "tests/smoke_v6.py"):
    assert (CODE / f).exists(), f"{f} missing: dataset does not hold knee_pipeline_v6"
print("backbone:", COAT, "| GPU0 lr 4e-5 | GPU1 lr 2e-5")

backbone: coatnet_rmlp_1_rw2_224.sw_in12k_ft_in1k | GPU0 lr 4e-5 | GPU1 lr 2e-5


In [2]:
# tests: if any fails, the assert stops the notebook BEFORE training starts
for t in ("tests/test_geom.py", "tests/smoke_v6.py"):
    r = subprocess.run([sys.executable, str(CODE / t)], capture_output=True, text=True, cwd=str(CODE))
    print(f"== {t} (exit {r.returncode})\n{r.stdout[-800:]}")
    assert r.returncode == 0, f"{t} failed:\n{r.stderr[-1500:]}"
EXTRA = ["--head", "tokens"]
print("extra train args:", EXTRA)
# precheck: CoAtNet at 160 px WITH pretrained weights (needs internet) -> forward + backward on CPU; fails fast, before any GPU time
chk = r"""
import torch, sys
from model import KneeNet
m = KneeNet(sys.argv[1], pretrained=True, head="tokens")
x = torch.rand(2, 3, 160, 160) * 255
f = m._enc((x - m.mean) / m.std); f.float().mean().backward()
print("coatnet precheck OK | feature dim", tuple(f.shape), "| vit_size", m.vit_size, "| params %.1fM" % (sum(p.numel() for p in m.parameters()) / 1e6))
if torch.cuda.is_available():
    m = m.cuda().train(); m.zero_grad(set_to_none=True); torch.cuda.reset_peak_memory_stats()
    x = torch.rand(96, 3, 160, 160, device="cuda") * 255          # bs 2 x 3 series x 16 slices = 96 windows
    with torch.autocast("cuda"):
        f = m._enc((x - m.mean) / m.std)
    f.float().mean().backward(); peak = torch.cuda.max_memory_allocated() / 1e9
    print("GPU probe (96 windows, bs 2): peak %.1f GB of ~14.5" % peak)
    assert peak < 12.5, "too much GPU memory for a T4 at bs 2: use --bs 1 --accum 8"
"""
r = subprocess.run([sys.executable, "-c", chk, COAT], capture_output=True, text=True, cwd=str(CODE))
print(r.stdout[-600:]); assert r.returncode == 0, "CoAtNet precheck failed:\n" + r.stderr[-2500:]

== tests/test_geom.py (exit 0)
canonical orientation identical for 24 acquisition conventions
right/left mirror normalisation OK for 3 planes
laterality rules OK
ALL GEOM TESTS PASS

== tests/smoke_v6.py (exit 0)
head=series: forward/backward OK, absent-series masking OK, params 5.69M
head=tokens: forward/backward OK, absent-series masking OK, params 5.18M
backbone convnext_tiny.in12k_ft_in1k: OK | vit_size None | params 28.9M
backbone efficientnet_b2: OK | vit_size None | params 8.9M
backbone vit_small_patch14_dinov2.lvd142m: OK | vit_size 168 | params 22.5M
backbone vit_small_patch16_224.dino: OK | vit_size 160 | params 22.6M
backbone deit_tiny_patch16_224.fb_in1k: OK | vit_size 160 | params 6.4M
v5-style state_dict loads into head='series'
KneeDS geometry parity OK
ALL V6 SMOKE TESTS PASS

extra train args: ['--head', 'tokens']
coatnet precheck OK | feature dim (2, 768) | vit_size 160 | params 42.0M
GPU probe (96 windows, bs 2): peak 7.9 GB of ~14.5



In [3]:
BASE_ARGS = ["--cache_dir", str(CACHE), "--labels", str(LABELS), "--epochs", "12", "--pretrained",
             "--workers", "2", "--train_slices", "16", "--bs", "2", "--accum", "4"]
PLAN = {0: [("v6coat_lr4e5_f0", ["--backbone", COAT, "--lr_backbone", "4e-5"])],
        1: [("v6coat_lr2e5_f0", ["--backbone", COAT, "--lr_backbone", "2e-5"])]}
NAMES = [n for jobs in PLAN.values() for n, _ in jobs]
FIRST_EPOCH_S, STALL_S, EPOCH_BUDGET_S = 45 * 60, 45 * 60, 1100
LOGS, RC = {}, {}
def n_epochs(f): return sum("ep " in l for l in f.read_text().splitlines()) if f.exists() else 0
def run_one(gpu, name, jx):
    out = W / name
    if (out / "oof.csv").exists():
        print(f"[{name}] oof.csv exists, skipped"); RC[name] = 0; return
    cmd = [sys.executable, str(CODE / "train.py")] + BASE_ARGS + EXTRA + jx + ["--only_folds", "0", "--out", str(out)]
    LOGS[name] = W / f"{name}.log"
    print(f"[{name}] GPU {gpu}: {' '.join(cmd)}", flush=True)
    fh = open(LOGS[name], "w")
    p = subprocess.Popen(cmd, stdout=fh, stderr=subprocess.STDOUT, env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True"), cwd=str(CODE))
    t0 = last_t = time.time(); seen = 0
    while p.poll() is None:
        time.sleep(30)
        k = n_epochs(LOGS[name])
        if k > seen:
            seen, last_t = k, time.time()
            m = re.search(r"\| (\d+)s\s*$", [l for l in LOGS[name].read_text().splitlines() if "ep " in l][-1])
            if m and int(m.group(1)) > EPOCH_BUDGET_S:
                p.kill(); p.wait(); RC[name] = f"KILLED: epoch took {m.group(1)}s (> {EPOCH_BUDGET_S}s budget)"; print(f"[{name}] {RC[name]}", flush=True); fh.close(); return
        limit = FIRST_EPOCH_S if seen == 0 else STALL_S
        if time.time() - (t0 if seen == 0 else last_t) > limit:
            p.kill(); p.wait(); RC[name] = "KILLED by watchdog (no progress)"; print(f"[{name}] {RC[name]}", flush=True); fh.close(); return
    fh.close(); RC[name] = p.returncode
    print(f"[{name}] exit code {RC[name]}" + (" (killed by the OS: probably out of RAM)" if RC[name] == -9 else ""), flush=True)
def run_gpu(gpu, jobs):
    for name, jx in jobs: run_one(gpu, name, jx)
threads = [threading.Thread(target=run_gpu, args=(g, jobs)) for g, jobs in PLAN.items()]
for th in threads: th.start()
t0 = time.time()
while any(th.is_alive() for th in threads):
    time.sleep(600)
    last = {}
    for n in NAMES:
        f = LOGS.get(n)
        ls = [l for l in f.read_text().splitlines() if "ep " in l] if f is not None and f.exists() else []
        last[n] = ls[-1] if ls else "(not started or no epoch yet)"
    print(f"[{(time.time()-t0)/3600:.2f} h] {json.dumps(last, indent=1)}", flush=True)
for th in threads: th.join()
print("all jobs finished | results:", RC)
for n, rc in RC.items():
    if rc != 0:
        print(f"--- {n} FAILED, log tail ---"); print("\n".join(LOGS[n].read_text().splitlines()[-25:]))

[v6coat_lr4e5_f0] GPU 0: /usr/bin/python3 /kaggle/input/datasets/theokorir/knee-mri-pipeline/knee_pipeline/train.py --cache_dir /kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/cache --labels /kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/labels_soft.csv --epochs 12 --pretrained --workers 2 --train_slices 16 --bs 2 --accum 4 --head tokens --backbone coatnet_rmlp_1_rw2_224.sw_in12k_ft_in1k --lr_backbone 4e-5 --only_folds 0 --out /kaggle/working/v6coat_lr4e5_f0
[v6coat_lr2e5_f0] GPU 1: /usr/bin/python3 /kaggle/input/datasets/theokorir/knee-mri-pipeline/knee_pipeline/train.py --cache_dir /kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/cache --labels /kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/labels_soft.csv --epochs 12 --pretrained --workers 2 --train_slices 16 --bs 2 --accum 4 --head tokens --backbone coatnet_rmlp_1_rw2_224.sw_in12k_ft_in1k --lr_backbone 2e-5 --only_folds 0 --out /kaggle/working/v6coat_lr2e5_f0
[0.17 h] {
 "v6coat_

In [4]:
# ---- fold-0 comparison on the FIXED old labels (confident: y>=0.9 or <=0.1) ----
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata
LAB = ["ACL","MCL","Medial Meniscus","Lateral Meniscus","Medial OA","Lateral OA","PF OA","Effusion","Synovitis","Baker's","Contusion","Fracture"]
KEY4 = LAB[:4]
base = pd.read_csv(BASE)
Y = {l: base[f"y_{l}"].values for l in LAB}
def auc_l(y, p):
    m = (y >= 0.9) | (y <= 0.1)
    return roc_auc_score((y[m] >= 0.9).astype(int), p[m]) if m.sum() > 20 and len({*(y[m] >= 0.9)}) == 2 else np.nan
PRED = {"baseline": base}
for name in NAMES:
    f = W / name / "oof.csv"
    if f.exists():
        d = pd.read_csv(f); assert len(d) == len(base), f"{name}: row count differs from baseline"
        PRED[name.replace("_f0", "")] = d
cx = [p for p in Path("/kaggle/input").rglob("oof_v6convnext.csv")]
if cx: PRED["convnext"] = pd.read_csv(cx[0]); print("convnext OOF found:", cx[0])
else: print("convnext OOF not attached: skipping the convnext comparison")
assert any(k not in ("baseline", "convnext") for k in PRED), "no DINO oof.csv found: the training jobs failed, read the log tails printed by the training cell"
ref = next(v for k, v in PRED.items() if k not in ("baseline", "convnext"))
mask = ref[f"p_{LAB[0]}"].notna().values            # fold-0 validation studies = the rows this screen predicted
print("fold-0 validation studies:", int(mask.sum()))
RK = {n: {l: rankdata(d.loc[mask, f"p_{l}"].values) / mask.sum() for l in LAB} for n, d in PRED.items()}
names = [n for n in RK if n not in ("baseline", "convnext")]
if "convnext" in RK:
    for n in names: RK[f"ens {n}+convnext"] = {l: (RK[n][l] + RK["convnext"][l]) / 2 for l in LAB}
T = pd.DataFrame({n: {l: auc_l(Y[l][mask], r[l]) for l in LAB} for n, r in RK.items()})
T.loc["MACRO"] = T.loc[LAB].mean(); T.loc["KEY-4"] = T.loc[KEY4].mean()
print("\nFold 0 only (ranks within fold):\n", T.round(3).to_string())
print("\nreference from the ConvNeXt run: fold-0 raw macro 0.8328, baseline 0.7795. Seed noise ~0.01.")
print("rule: worth a 5-fold only if an arm beats ConvNeXt-T fold 0 (0.833 macro / 0.835 KEY-4) by more than ~0.015, OR its ensemble row beats ConvNeXt alone by that margin (different families may add what seeds did not). 5-fold cost per GPU = epoch seconds x 12 x 3 folds.")

convnext OOF found: /kaggle/input/notebooks/theokorir/knee-v6-convnext-5fold/oof_v6convnext.csv
fold-0 validation studies: 882

Fold 0 only (ranks within fold):
                   baseline  v6coat_lr2e5  convnext  ens v6coat_lr2e5+convnext
ACL                  0.833         0.895     0.892                      0.902
MCL                  0.713         0.775     0.805                      0.798
Medial Meniscus      0.757         0.825     0.870                      0.855
Lateral Meniscus     0.715         0.740     0.776                      0.764
Medial OA            0.861         0.865     0.874                      0.874
Lateral OA           0.749         0.755     0.767                      0.766
PF OA                0.722         0.732     0.740                      0.741
Effusion             0.801         0.826     0.831                      0.833
Synovitis            0.799         0.782     0.789                      0.788
Baker's              0.747         0.848     0.865        